# CovCollExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/CovCollExamples.md`


Notebook source link: [CovCollExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/CovCollExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "CovCollExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")


In [ ]:
# Signal/History workflow: explore covariates, spikes, history design, and events.
time = np.linspace(0.0, 4.0, 4001)
s1 = np.sin(2.0 * np.pi * 1.2 * time)
s2 = 0.5 * np.cos(2.0 * np.pi * 0.45 * time + 0.4)
s3 = s1 + s2

cov = Covariate(time=time, data=np.column_stack([s1, s2, s3]), name="signals", labels=["s1", "s2", "s3"])
base_prob = np.clip(0.005 + 0.03 * (s3 > np.percentile(s3, 65)), 0.0, 0.4)
spike_times = time[rng.random(time.size) < base_prob]
spikes = SpikeTrain(spike_times=spike_times, t_start=float(time[0]), t_end=float(time[-1]), name="unit_1")

history = HistoryBasis(np.array([0.0, 0.005, 0.010, 0.020, 0.050]))
sample_times = time[::20]
H = history.design_matrix(spikes.spike_times, sample_times)

burst_events = Events(times=np.array([0.5, 1.6, 2.4, 3.2]), labels=["A", "B", "C", "D"])
centers, counts = spikes.bin_counts(bin_size_s=0.02)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
axes[0].plot(time, cov.data[:, 0], label="s1", linewidth=1.0)
axes[0].plot(time, cov.data[:, 1], label="s2", linewidth=1.0)
axes[0].plot(time, cov.data[:, 2], label="s3", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: signal and covariates")
axes[0].legend(loc="upper right")

axes[1].bar(centers, counts, width=0.018, color="tab:gray")
axes[1].vlines(burst_events.times, ymin=0.0, ymax=max(counts.max(), 1.0), color="tab:red", linewidth=1.0)
axes[1].set_title("Binned spikes with event markers")
axes[1].set_ylabel("count/bin")

im = axes[2].imshow(H.T, aspect="auto", origin="lower", cmap="magma")
axes[2].set_title("History basis design matrix")
axes[2].set_xlabel("time index")
axes[2].set_ylabel("history bin")
fig.colorbar(im, ax=axes[2], fraction=0.035, pad=0.02)

plt.tight_layout()
plt.show()

assert H.ndim == 2 and H.shape[1] == history.n_bins
assert spikes.spike_times.size > 5, "Not enough spikes generated"


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
